# 01.01 Systems of Linear Equations


## Notes
- Add theory, derivations, examples, and solved exercises here.
- Add references at the end.


In [ ]:
import numpy as np

# ------------------------------------------------------------
# Utilities (no np.linalg.*)
# ------------------------------------------------------------
def print_aug(A, b, title="Augmented matrix [A|b]"):
    """Pretty-print the augmented matrix."""
    Ab = np.hstack([A.astype(float), b.astype(float).reshape(-1, 1)])
    print("\n" + title)
    for i in range(Ab.shape[0]):
        row = Ab[i]
        left = "  ".join([f"{v:8.3f}" for v in row[:-1]])
        right = f"{row[-1]:8.3f}"
        print(f"[ {left} | {right} ]")

def swap_rows(M, i, j):
    """Swap rows i and j in-place."""
    if i != j:
        tmp = M[i].copy()
        M[i] = M[j]
        M[j] = tmp

def gaussian_elimination(A, b, eps=1e-12, verbose=True):
    """
    Row-reduce augmented matrix [A|b] to (row) echelon form using
    partial pivoting. Returns:
      U (upper-ish), y (transformed RHS), pivots (list of pivot columns)
    """
    A = A.astype(float).copy()
    b = b.astype(float).reshape(-1, 1).copy()

    m, n = A.shape
    pivots = []
    row = 0

    if verbose:
        print_aug(A, b, "Start")

    for col in range(n):
        if row >= m:
            break

        # Find pivot row with largest abs value in this column from current row downward
        pivot_row = row
        max_val = abs(A[row, col])
        for r in range(row + 1, m):
            v = abs(A[r, col])
            if v > max_val:
                max_val = v
                pivot_row = r

        # If column is (numerically) zero below row, skip this column updated 

        if max_val < eps:
            continue

        # Swap pivot row into position
        swap_rows(A, row, pivot_row)
        swap_rows(b, row, pivot_row)

        pivots.append(col)

        # Eliminate entries below pivot
        pivot = A[row, col]
        for r in range(row + 1, m):
            factor = A[r, col] / pivot
            # Row operation: Rr <- Rr - factor * Rrow
            A[r, col:] = A[r, col:] - factor * A[row, col:]
            b[r, 0] = b[r, 0] - factor * b[row, 0]

        if verbose:
            print_aug(A, b, f"After eliminating column {col} (pivot at row {row})")

        row += 1

    return A, b, pivots

def classify_system(U, y, pivots, eps=1e-10):
    """
    Classify:
      - 'inconsistent' (no solution)
      - 'unique'
      - 'infinite'
    based on echelon form.
    """
    m, n = U.shape
    rankA = 0
    rankAb = 0

    # Count nonzero rows in U and in augmented [U|y] without using np.linalg.rank
    for i in range(m):
        rowA_nonzero = False
        rowAb_nonzero = False

        # Check A-part
        for j in range(n):
            if abs(U[i, j]) > eps:
                rowA_nonzero = True
                break

        # Check augmented part
        if rowA_nonzero:
            rowAb_nonzero = True
        else:
            if abs(y[i, 0]) > eps:
                rowAb_nonzero = True

        if rowA_nonzero:
            rankA += 1
        if rowAb_nonzero:
            rankAb += 1

    if rankAb > rankA:
        return "inconsistent", rankA, rankAb  # no solution

    # If rank equals number of unknowns => unique solution
    if rankA == n:
        return "unique", rankA, rankAb

    # Otherwise infinite solutions (free variables exist)
    return "infinite", rankA, rankAb

def back_substitution(U, y, pivots, eps=1e-12):
    """
    Solve Ux = y for x when the solution is unique (assumes n pivots).
    U is in echelon/upper-triangular-like form.
    """
    m, n = U.shape
    x = np.zeros((n, 1), dtype=float)

    # We assume pivots are in increasing order and there are n of them for unique solution
    # Solve from last pivot row to first pivot row
    num_piv = len(pivots)
    for k in range(num_piv - 1, -1, -1):
        pivot_col = pivots[k]
        pivot_row = k  # because we advanced row by 1 per pivot in elimination

        # Compute sum_{j>pivot_col} U[pivot_row, j] * x[j]
        s = 0.0
        for j in range(pivot_col + 1, n):
            s += U[pivot_row, j] * x[j, 0]

        denom = U[pivot_row, pivot_col]
        if abs(denom) < eps:
            raise ValueError("Zero pivot encountered in back substitution.")

        x[pivot_col, 0] = (y[pivot_row, 0] - s) / denom

    return x

def solve_linear_system(A, b, verbose=True):
    """
    Full pipeline:
      1) Gaussian elimination
      2) classify system
      3) solve if unique; show structure if infinite; explain if inconsistent
    """
    U, y, pivots = gaussian_elimination(A, b, verbose=verbose)
    status, rankA, rankAb = classify_system(U, y, pivots)

    print("\nClassification:")
    print(f"  rank(A)   = {rankA}")
    print(f"  rank([A|b]) = {rankAb}")
    print(f"  status    = {status}")

    if status == "inconsistent":
        print("\nNo solution: a row became [0 0 ... 0 | nonzero].")
        return None

    if status == "unique":
        x = back_substitution(U, y, pivots)
        print("\nUnique solution x:")
        print(x.reshape(-1))
        return x

    # Infinite solutions
    print("\nInfinitely many solutions: there are free variables.")
    # Provide one particular solution by setting free vars to 0 (simple demonstration)
    n = A.shape[1]
    x = np.zeros((n, 1), dtype=float)
    # Solve pivot variables in reverse with free vars = 0
    for k in range(len(pivots) - 1, -1, -1):
        pivot_col = pivots[k]
        pivot_row = k
        s = 0.0
        for j in range(pivot_col + 1, n):
            s += U[pivot_row, j] * x[j, 0]
        x[pivot_col, 0] = (y[pivot_row, 0] - s) / U[pivot_row, pivot_col]

    print("\nOne particular solution (free vars set to 0):")
    print(x.reshape(-1))
    return x


# ------------------------------------------------------------
# Examples (matching the PDF section)
# ------------------------------------------------------------

# Example 2.4 from the page: no solution
A1 = np.array([
    [1,  1, 1],
    [1, -1, 2],
    [2,  0, 3]
], dtype=float)
b1 = np.array([3, 2, 1], dtype=float)

print("\n============================")
print("Example: NO SOLUTION (like Eq. 2.4)")
print("============================")
solve_linear_system(A1, b1, verbose=True)

# Example 2.5: unique solution (1,1,1)
A2 = np.array([
    [1,  1, 1],
    [1, -1, 2],
    [0,  1, 1]
], dtype=float)
b2 = np.array([3, 2, 2], dtype=float)

print("\n============================")
print("Example: UNIQUE SOLUTION (like Eq. 2.5)")
print("============================")
solve_linear_system(A2, b2, verbose=True)

# Example 2.6: infinite solutions (third equation redundant)
A3 = np.array([
    [1,  1, 1],
    [1, -1, 2],
    [2,  0, 3]
], dtype=float)
b3 = np.array([3, 2, 5], dtype=float)

print("\n============================")
print("Example: INFINITE SOLUTIONS (like Eq. 2.6)")
print("============================")
solve_linear_system(A3, b3, verbose=True)

print("============================")

print("============================")


print("============================")


Example: NO SOLUTION (like Eq. 2.4)

Start
[    1.000     1.000     1.000 |    3.000 ]
[    1.000    -1.000     2.000 |    2.000 ]
[    2.000     0.000     3.000 |    1.000 ]

After eliminating column 0 (pivot at row 0)
[    2.000     0.000     3.000 |    1.000 ]
[    0.000    -1.000     0.500 |    1.500 ]
[    0.000     1.000    -0.500 |    2.500 ]

After eliminating column 1 (pivot at row 1)
[    2.000     0.000     3.000 |    1.000 ]
[    0.000    -1.000     0.500 |    1.500 ]
[    0.000     0.000     0.000 |    4.000 ]

Classification:
  rank(A)   = 2
  rank([A|b]) = 3
  status    = inconsistent

No solution: a row became [0 0 ... 0 | nonzero].

Example: UNIQUE SOLUTION (like Eq. 2.5)

Start
[    1.000     1.000     1.000 |    3.000 ]
[    1.000    -1.000     2.000 |    2.000 ]
[    0.000     1.000     1.000 |    2.000 ]

After eliminating column 0 (pivot at row 0)
[    1.000     1.000     1.000 |    3.000 ]
[    0.000    -2.000     1.000 |   -1.000 ]
[    0.000     1.000     1.00

array([[2.5],
       [0.5],
       [0. ]])